In [1]:
import os
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from src.models import DinoExtractor

# ── Config ────────────────────────────────────────────────────────────────────
FILE_PATH  = "data/processed/dataset/train/negatives/neg_0_M108447638RC.npy"
IMG_SIZE   = 224
PATCH_SIZE = 16
GRID_SIZE  = IMG_SIZE // PATCH_SIZE  # 14

/Users/finnhertsch/projects/luna_hole/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── I/O ───────────────────────────────────────────────────────────────────────
def resolve_path(path: str) -> str:
    if os.path.exists(path):
        return path
    target = os.path.basename(path)
    for root, _, files in os.walk("data"):
        if target in files:
            found = os.path.join(root, target)
            print(f"Found at: {found}")
            return found
    raise FileNotFoundError(f"{target} not found under data/")


def load_tile(path: str, device: torch.device) -> tuple[np.ndarray, torch.Tensor]:
    raw = np.load(path).astype(np.float32)
    img = (raw - raw.min()) / (raw.max() - raw.min())

    tensor = (
        torch.from_numpy(img)
        .unsqueeze(0).unsqueeze(0)          # [1, 1, H, W]
        .repeat(1, 3, 1, 1)                  # [1, 3, H, W]
        .to(device)
    )
    tensor = F.interpolate(tensor, size=(IMG_SIZE, IMG_SIZE), mode="bicubic")
    return img, tensor

In [3]:
# ── Attention extraction ───────────────────────────────────────────────────────
def _build_patched_forward(attn_module, captured: dict):
    def forward(x: torch.Tensor, rope=None) -> torch.Tensor:
        B, N, C = x.shape
        head_dim = C // attn_module.num_heads

        qkv = (
            attn_module.qkv(x)
            .reshape(B, N, 3, attn_module.num_heads, head_dim)
            .permute(2, 0, 3, 1, 4)
        )
        q, k, v = qkv.unbind(0)

        # RoPE auf q und k anwenden falls vorhanden
        if rope is not None:
            from timm.layers import apply_rot_embed_cat
            q = apply_rot_embed_cat(q, rope)
            k = apply_rot_embed_cat(k, rope)

        attn = (q @ k.transpose(-2, -1)) * attn_module.scale
        attn = attn.softmax(dim=-1)
        captured["attn"] = attn.detach()

        attn = attn_module.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = attn_module.proj(x)
        x = attn_module.proj_drop(x)
        return x

    return forward

In [ ]:
# ── Attention extraction ───────────────────────────────────────────────────────

def _rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def _apply_rope(x: torch.Tensor, rope: tuple) -> torch.Tensor:
    """Standard RoPE: rope = (sin, cos) tuple aus DINOv3."""
    sin, cos = rope
    return x * cos + _rotate_half(x) * sin


def _build_patched_forward(attn_module, captured: dict):
    def forward(x: torch.Tensor, rope=None) -> torch.Tensor:
        B, N, C = x.shape
        head_dim = C // attn_module.num_heads

        qkv = (
            attn_module.qkv(x)
            .reshape(B, N, 3, attn_module.num_heads, head_dim)
            .permute(2, 0, 3, 1, 4)
        )
        q, k, v = qkv.unbind(0)

        if rope is not None:
            q = _apply_rope(q, rope)
            k = _apply_rope(k, rope)

        attn = (q * attn_module.scale) @ k.transpose(-2, -1)
        attn = attn.softmax(dim=-1)
        captured["attn"] = attn.detach()          # ← hier capturen, vor dropout

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = attn_module.proj(x)
        return x

    return forward


def extract_last_block_attention(model, img_tensor: torch.Tensor) -> torch.Tensor:
    base      = model.model.get_base_model()
    last_attn = base.blocks[-1].attn
    captured: dict = {}

    original_forward  = last_attn.forward
    last_attn.forward = _build_patched_forward(last_attn, captured)

    try:
        with torch.no_grad():
            model(img_tensor)
    finally:
        last_attn.forward = original_forward

    if "attn" not in captured:
        raise RuntimeError("Patched forward nie aufgerufen – Modellstruktur prüfen.")

    return captured["attn"]  # [B, heads, tokens, tokens]


def cls_attention_maps(attn: torch.Tensor, grid: int) -> torch.Tensor:
    """CLS-token → patch attention per head, reshaped to spatial grid."""
    return attn[0, :, 0, 1:].reshape(-1, grid, grid)  # [heads, H, W]


# ── Visualisation ─────────────────────────────────────────────────────────────
def show_attention(image: np.ndarray, maps: torch.Tensor, upsample: int = 256) -> None:
    nh   = maps.shape[0]
    fig, axes = plt.subplots(1, nh + 1, figsize=(4 * (nh + 1), 4))

    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("NAC Tile")
    axes[0].axis("off")

    for i, ax in enumerate(axes[1:]):
        a = maps[i].cpu().numpy()
        a = np.array(Image.fromarray(a).resize((upsample, upsample), resample=Image.BICUBIC))
        ax.imshow(a, cmap="magma")
        ax.set_title(f"Head {i + 1}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# ── Entry point ───────────────────────────────────────────────────────────────

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()

weights = "models/meta/dinov3/dinov3_vits16_pretrain_lvd.pth"
adapter_path = "models/best_lora_pit_model/best_lora_pit_model_qkv_proj"

model = DinoExtractor(weights_path=weights, model_size='vits16')
model.model.load_adapter(adapter_path, adapter_name="trained_pit_adapter")
model.model.set_adapter("trained_pit_adapter")
model.to(device)
model.eval()

path        = resolve_path(FILE_PATH)
image, tensor = load_tile(path, device)
raw_attn    = extract_last_block_attention(model, tensor)
maps        = cls_attention_maps(raw_attn, GRID_SIZE)
show_attention(image, maps)

2026-05-11 15:30:56 | INFO     | src.models | Load DINO model: dino_vits16_lvd_vits16...


Using cache found in /Users/finnhertsch/.cache/torch/hub/facebookresearch_dinov3_main


trainable params: 2,396,160 || all params: 23,997,312 || trainable%: 9.9851
Found at: data/processed/dataset/test/negatives/neg_0_M108447638RC.npy


RuntimeError: attn_drop wurde nie aufgerufen – fused_attn ist möglicherweise noch aktiv oder wird durch den Adapter überschrieben.